#### This is exploratory data analysis of our processed openAQ data from S3. We will explore the data behaviour and wil make some decision for our further analysis

## Raw data summary.

we will see raw hourly and daily data. We did not add weather data yet. we will later include weather data like temparature, wind speed, cloud coverage later to enchance our analysis

In [ ]:

# Dataset overview: OpenAQ hourly files


from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path(".")

openaq_by_city_dir = project_root / "data" / "processed" / "openaq_by_city"
enriched_by_city_dir = project_root / "data" / "processed" / "enriched" / "by_city"
enriched_combined_dir = project_root / "data" / "processed" / "enriched" / "combined"

summary_output_dir = project_root / "data" / "processed" / "dataset_overview"
summary_output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Helper functions


def count_csv_rows(path):
    # Counts rows in a CSV without loading the full file into memory
    count = 0

    for chunk in pd.read_csv(path, chunksize=250_000):
        count += len(chunk)

    return count


def get_table_shape_and_head(path, head_rows=5):
    # Returns shape and first few rows for CSV or Parquet
    path = Path(path)

    if path.suffix == ".csv":
        n_rows = count_csv_rows(path)
        head = pd.read_csv(path, nrows=head_rows)
        n_cols = len(head.columns)

    elif path.suffix == ".parquet":
        df_head = pd.read_parquet(path)
        n_rows, n_cols = df_head.shape
        head = df_head.head(head_rows)

    else:
        raise ValueError(f"Unsupported file type: {path}")

    return {
        "path": str(path),
        "rows": n_rows,
        "columns": n_cols,
        "head": head,
    }


def summarize_city_files(root_dir, file_pattern):
    # Summarizes matching city-level files
    files = sorted(Path(root_dir).glob(f"*/*{file_pattern}"))

    rows = []

    for path in files:
        city_slug = path.parent.name

        info = get_table_shape_and_head(path, head_rows=5)

        rows.append(
            {
                "city_slug": city_slug,
                "file_name": path.name,
                "path": str(path),
                "rows": info["rows"],
                "columns": info["columns"],
                "size_mb": path.stat().st_size / 1024 / 1024,
            }
        )

    summary = pd.DataFrame(rows)

    if summary.empty:
        return summary, pd.DataFrame()

    total_row = pd.DataFrame(
        [
            {
                "city_slug": "TOTAL",
                "file_name": file_pattern,
                "path": str(root_dir),
                "rows": summary["rows"].sum(),
                "columns": np.nan,
                "size_mb": summary["size_mb"].sum(),
            }
        ]
    )

    summary_with_total = pd.concat([summary, total_row], ignore_index=True)

    first_file_head = (
        pd.read_csv(files[0], nrows=5)
        if files[0].suffix == ".csv"
        else pd.read_parquet(files[0]).head(5)
    )

    return summary_with_total, first_file_head

In [9]:
# ============================================================
# OpenAQ location-hourly files
# One row = one monitoring location, one UTC hour
# ============================================================

location_hourly_summary, location_hourly_head = summarize_city_files(
    root_dir=openaq_by_city_dir,
    file_pattern="_location_hourly.csv",
)

location_hourly_summary.to_csv(
    summary_output_dir / "location_hourly_file_summary.csv",
    index=False,
)

location_hourly_summary

,city_slug,file_name,path,rows,columns,size_mb
0,abu_dhabi,abu_dhabi_location_hourly.csv,data\processed\openaq_by_city\abu_dhabi\abu_dh...,8411,26.0,1.237903
1,bangkok,bangkok_location_hourly.csv,data\processed\openaq_by_city\bangkok\bangkok_...,92870,26.0,15.870592
2,beijing,beijing_location_hourly.csv,data\processed\openaq_by_city\beijing\beijing_...,35042,26.0,7.740582
3,cape_town,cape_town_location_hourly.csv,data\processed\openaq_by_city\cape_town\cape_t...,23931,26.0,5.116937
4,delhi,delhi_location_hourly.csv,data\processed\openaq_by_city\delhi\delhi_loca...,40121,26.0,6.902556
5,dhaka,dhaka_location_hourly.csv,data\processed\openaq_by_city\dhaka\dhaka_loca...,6484,26.0,1.486906
6,dubai,dubai_location_hourly.csv,data\processed\openaq_by_city\dubai\dubai_loca...,50994,26.0,8.377801
7,jakarta,jakarta_location_hourly.csv,data\processed\openaq_by_city\jakarta\jakarta_...,25978,26.0,4.173594
8,kuwait_city,kuwait_city_location_hourly.csv,data\processed\openaq_by_city\kuwait_city\kuwa...,38454,26.0,5.470056
9,lahore,lahore_location_hourly.csv,data\processed\openaq_by_city\lahore\lahore_lo...,16279,26.0,2.388899


In [10]:
location_hourly_head

,city,country,iso,city_group,location_id,location_name,timezone,lat,lon,datetime_hour_utc,...,pm25,pm10,pm25_raw_measurements,pm10_raw_measurements,pm25_sensor_count_hourly,pm10_sensor_count_hourly,has_pm25,has_pm10,has_pm25_pm10_pair,pm25_pm10_ratio
0,abu_dhabi,NaN,NaN,NaN,1285339,SPARTAN - Abu Dhabi,NaN,24.4416,54.6166,2020-01-01 13:00:00+00,...,18.7,NaN,1,NaN,1,NaN,True,False,False,NaN
1,abu_dhabi,NaN,NaN,NaN,1285339,SPARTAN - Abu Dhabi,NaN,24.4416,54.6166,2020-01-01 20:00:00+00,...,29.6,NaN,1,NaN,1,NaN,True,False,False,NaN
2,abu_dhabi,NaN,NaN,NaN,1285339,SPARTAN - Abu Dhabi,NaN,24.4416,54.6166,2020-01-02 00:00:00+00,...,28.7,NaN,1,NaN,1,NaN,True,False,False,NaN
3,abu_dhabi,NaN,NaN,NaN,1285339,SPARTAN - Abu Dhabi,NaN,24.4416,54.6166,2020-01-02 10:00:00+00,...,50.5,NaN,1,NaN,1,NaN,True,False,False,NaN
4,abu_dhabi,NaN,NaN,NaN,1285339,SPARTAN - Abu Dhabi,NaN,24.4416,54.6166,2020-01-03 08:00:00+00,...,56.0,NaN,1,NaN,1,NaN,True,False,False,NaN



#### OpenAQ city-hourly files
#### One row = one city, one UTC hour, aggregated across selected locations


In [11]:


city_hourly_summary, city_hourly_head = summarize_city_files(
    root_dir=openaq_by_city_dir,
    file_pattern="_city_hourly.csv",
)

city_hourly_summary.to_csv(
    summary_output_dir / "city_hourly_file_summary.csv",
    index=False,
)

city_hourly_summary

,city_slug,file_name,path,rows,columns,size_mb
0,abu_dhabi,abu_dhabi_city_hourly.csv,data\processed\openaq_by_city\abu_dhabi\abu_dh...,8411,29.0,1.071799
1,bangkok,bangkok_city_hourly.csv,data\processed\openaq_by_city\bangkok\bangkok_...,33587,29.0,7.028553
2,beijing,beijing_city_hourly.csv,data\processed\openaq_by_city\beijing\beijing_...,23293,29.0,6.431255
3,cape_town,cape_town_city_hourly.csv,data\processed\openaq_by_city\cape_town\cape_t...,11256,29.0,2.881158
4,delhi,delhi_city_hourly.csv,data\processed\openaq_by_city\delhi\delhi_city...,18659,29.0,4.041039
5,dhaka,dhaka_city_hourly.csv,data\processed\openaq_by_city\dhaka\dhaka_city...,6484,29.0,1.555721
6,dubai,dubai_city_hourly.csv,data\processed\openaq_by_city\dubai\dubai_city...,11682,29.0,2.922663
7,jakarta,jakarta_city_hourly.csv,data\processed\openaq_by_city\jakarta\jakarta_...,16557,29.0,4.127818
8,kuwait_city,kuwait_city_city_hourly.csv,data\processed\openaq_by_city\kuwait_city\kuwa...,38454,29.0,5.172091
9,lahore,lahore_city_hourly.csv,data\processed\openaq_by_city\lahore\lahore_ci...,15172,29.0,2.511588


In [1]:
# Combined city-level EDA before adding weather data

from pathlib import Path
import pandas as pd
import numpy as np

processed_root = Path("data/processed/openaq_by_city")
combined_output_dir = Path("data/processed/combined")
combined_output_dir.mkdir(parents=True, exist_ok=True)

# Find all city monthly files
city_monthly_files = sorted(processed_root.rglob("*_city_monthly.csv"))

print("City monthly files found:", len(city_monthly_files))

for path in city_monthly_files:
    print(path)

# Load and combine
monthly_frames = []

for path in city_monthly_files:
    df = pd.read_csv(path)
    df["source_file"] = str(path)
    monthly_frames.append(df)

combined_city_monthly = pd.concat(monthly_frames, ignore_index=True)

# Create a proper month date
combined_city_monthly["month_date"] = pd.to_datetime(
    combined_city_monthly["local_year"].astype(str)
    + "-"
    + combined_city_monthly["local_month"].astype(str).str.zfill(2)
    + "-01"
)

# Save combined monthly file
combined_city_monthly_path = combined_output_dir / "combined_city_monthly.csv"

combined_city_monthly.to_csv(
    combined_city_monthly_path,
    index=False
)

print("Saved combined monthly file:")
print(combined_city_monthly_path)

combined_city_monthly.head()

City monthly files found: 18
data\processed\openaq_by_city\abu_dhabi\abu_dhabi_city_monthly.csv
data\processed\openaq_by_city\bangkok\bangkok_city_monthly.csv
data\processed\openaq_by_city\beijing\beijing_city_monthly.csv
data\processed\openaq_by_city\cape_town\cape_town_city_monthly.csv
data\processed\openaq_by_city\delhi\delhi_city_monthly.csv
data\processed\openaq_by_city\dhaka\dhaka_city_monthly.csv
data\processed\openaq_by_city\dubai\dubai_city_monthly.csv
data\processed\openaq_by_city\jakarta\jakarta_city_monthly.csv
data\processed\openaq_by_city\kuwait_city\kuwait_city_city_monthly.csv
data\processed\openaq_by_city\lahore\lahore_city_monthly.csv
data\processed\openaq_by_city\london\london_city_monthly.csv
data\processed\openaq_by_city\los_angeles\los_angeles_city_monthly.csv
data\processed\openaq_by_city\mexico_city\mexico_city_city_monthly.csv
data\processed\openaq_by_city\new_york\new_york_city_monthly.csv
data\processed\openaq_by_city\riyadh\riyadh_city_monthly.csv
data\proce

,city,country,iso,city_group,local_year,local_month,valid_calendar_days_with_any_data,pm25_monthly_mean,pm25_monthly_median,pm25_monthly_p90,...,valid_pm10_locations_mean,days_in_month,pm25_month_coverage_pct,pm10_month_coverage_pct,pm25_month_is_usable,pm10_month_is_usable,pm25_exceedance_pct,pm10_exceedance_pct,source_file,month_date
0,abu_dhabi,NaN,NaN,NaN,2020,1,30,16.860079,15.377083,27.843482,...,0.0,31,96.774194,0.0,True,False,53.333333,NaN,data\processed\openaq_by_city\abu_dhabi\abu_dh...,2020-01-01
1,abu_dhabi,NaN,NaN,NaN,2020,3,31,25.895140,26.257143,35.504167,...,0.0,31,100.000000,0.0,True,False,93.548387,NaN,data\processed\openaq_by_city\abu_dhabi\abu_dh...,2020-03-01
2,abu_dhabi,NaN,NaN,NaN,2020,6,30,40.525951,37.244211,71.860720,...,0.0,30,100.000000,0.0,True,False,83.333333,NaN,data\processed\openaq_by_city\abu_dhabi\abu_dh...,2020-06-01
3,abu_dhabi,NaN,NaN,NaN,2022,10,31,42.953325,43.850000,74.868750,...,0.0,31,100.000000,0.0,True,False,83.870968,NaN,data\processed\openaq_by_city\abu_dhabi\abu_dh...,2022-10-01
4,abu_dhabi,NaN,NaN,NaN,2022,11,30,27.649935,23.968750,43.206667,...,0.0,30,100.000000,0.0,True,False,76.666667,NaN,data\processed\openaq_by_city\abu_dhabi\abu_dh...,2022-11-01


#### city level coverage and inclusion summary

In [2]:
# City-level coverage and inclusion summary

city_coverage_summary = (
    combined_city_monthly
    .groupby(["city", "country", "city_group"], dropna=False)
    .agg(
        first_month=("month_date", "min"),
        last_month=("month_date", "max"),
        months_with_any_data=("month_date", "nunique"),

        usable_pm25_months=("pm25_month_is_usable", lambda x: int(pd.Series(x).fillna(False).astype(bool).sum())),
        usable_pm10_months=("pm10_month_is_usable", lambda x: int(pd.Series(x).fillna(False).astype(bool).sum())),

        mean_pm25_month_coverage_pct=("pm25_month_coverage_pct", "mean"),
        median_pm25_month_coverage_pct=("pm25_month_coverage_pct", "median"),
        mean_pm10_month_coverage_pct=("pm10_month_coverage_pct", "mean"),
        median_pm10_month_coverage_pct=("pm10_month_coverage_pct", "median"),

        mean_pm25=("pm25_monthly_mean", "mean"),
        median_pm25=("pm25_monthly_median", "median"),
        p90_pm25=("pm25_monthly_p90", "mean"),

        mean_pm10=("pm10_monthly_mean", "mean"),
        median_pm10=("pm10_monthly_median", "median"),
        p90_pm10=("pm10_monthly_p90", "mean"),

        mean_pm25_pm10_ratio=("pm25_pm10_ratio_monthly_mean", "mean"),

        mean_active_locations=("active_locations_mean", "mean"),
        max_active_locations=("active_locations_max", "max"),

        pm25_exceedance_days=("pm25_exceedance_days", "sum"),
        pm10_exceedance_days=("pm10_exceedance_days", "sum"),

        mean_pm25_exceedance_pct=("pm25_exceedance_pct", "mean"),
        mean_pm10_exceedance_pct=("pm10_exceedance_pct", "mean"),
    )
    .reset_index()
)

# Selection rules
city_coverage_summary["core_city_pm25"] = (
    city_coverage_summary["usable_pm25_months"] >= 12
)

city_coverage_summary["strong_city_pm25"] = (
    city_coverage_summary["usable_pm25_months"] >= 18
)

city_coverage_summary["particle_profile_city"] = (
    city_coverage_summary["usable_pm10_months"] >= 6
)

city_coverage_summary["recommended_use"] = "Exclude for main analysis"

city_coverage_summary.loc[
    city_coverage_summary["usable_pm25_months"].between(6, 11),
    "recommended_use"
] = "Context / limited comparison"

city_coverage_summary.loc[
    city_coverage_summary["core_city_pm25"],
    "recommended_use"
] = "Core PM2.5 comparison"

city_coverage_summary.loc[
    city_coverage_summary["core_city_pm25"]
    & city_coverage_summary["particle_profile_city"],
    "recommended_use"
] = "Core + particle-profile analysis"

city_coverage_summary = city_coverage_summary.sort_values(
    ["usable_pm25_months", "usable_pm10_months", "mean_pm25_month_coverage_pct"],
    ascending=[False, False, False]
).reset_index(drop=True)

city_coverage_summary_path = combined_output_dir / "city_coverage_selection_summary.csv"

city_coverage_summary.to_csv(
    city_coverage_summary_path,
    index=False
)

print("Saved city coverage summary:")
print(city_coverage_summary_path)

city_coverage_summary[
    [
        "city",
        "first_month",
        "last_month",
        "months_with_any_data",
        "usable_pm25_months",
        "usable_pm10_months",
        "mean_pm25_month_coverage_pct",
        "mean_pm10_month_coverage_pct",
        "mean_pm25",
        "mean_pm10",
        "mean_pm25_pm10_ratio",
        "max_active_locations",
        "recommended_use"
    ]
]

Saved city coverage summary:
data\processed\combined\city_coverage_selection_summary.csv


,city,first_month,last_month,months_with_any_data,usable_pm25_months,usable_pm10_months,mean_pm25_month_coverage_pct,mean_pm10_month_coverage_pct,mean_pm25,mean_pm10,mean_pm25_pm10_ratio,max_active_locations,recommended_use
0,kuwait_city,2019-12-01,2025-03-01,62,57,0,89.605706,0.000000,-69.528859,NaN,NaN,1,Core PM2.5 comparison
1,bangkok,2021-04-01,2025-12-01,55,51,44,91.094680,82.060746,22.605649,45.483872,0.515329,5,Core + particle-profile analysis
2,london,2021-05-01,2026-01-01,55,50,50,87.235861,87.177210,10.783260,18.060735,0.643507,5,Core + particle-profile analysis
3,beijing,2020-01-01,2026-01-01,39,37,26,89.044468,61.846666,38.929807,62.773763,0.714165,3,Core + particle-profile analysis
4,singapore,2022-10-01,2026-01-01,40,37,4,88.700851,11.129032,15.140140,22.273790,0.860407,3,Core PM2.5 comparison
5,delhi,2019-12-01,2025-12-01,44,36,36,78.902039,78.902039,109.556047,214.367794,0.521209,3,Core + particle-profile analysis
6,tokyo,2023-07-01,2026-01-01,31,29,1,93.846209,3.225806,9.618451,13.956266,0.904774,5,Core PM2.5 comparison
7,los_angeles,2022-02-01,2025-12-01,34,28,7,82.019517,19.135267,11.015566,23.425332,0.804717,2,Core + particle-profile analysis
8,mexico_city,2020-10-01,2026-01-01,33,27,27,76.321277,75.044221,20.725139,42.738565,0.511866,4,Core + particle-profile analysis
9,lahore,2023-11-01,2025-12-01,26,25,5,96.282051,18.717949,104.672869,304.438779,0.499836,4,Core PM2.5 comparison


#### # Month-level availability: how many cities are usable in each month?

In [3]:


month_availability = (
    combined_city_monthly.groupby("month_date")
    .agg(
        cities_with_any_data=("city", "nunique"),
        usable_pm25_cities=(
            "pm25_month_is_usable",
            lambda x: int(pd.Series(x).fillna(False).astype(bool).sum()),
        ),
        usable_pm10_cities=(
            "pm10_month_is_usable",
            lambda x: int(pd.Series(x).fillna(False).astype(bool).sum()),
        ),
    )
    .reset_index()
    .sort_values("month_date")
)

month_availability_path = combined_output_dir / "month_availability_summary.csv"

month_availability.to_csv(month_availability_path, index=False)

print("Saved month availability summary:")
print(month_availability_path)

month_availability.tail(30)

Saved month availability summary:
data\processed\combined\month_availability_summary.csv


,month_date,cities_with_any_data,usable_pm25_cities,usable_pm10_cities
44,2023-08-01,7,7,4
45,2023-09-01,7,7,4
46,2023-10-01,7,5,3
47,2023-11-01,9,7,4
48,2023-12-01,10,8,4
49,2024-01-01,10,10,4
50,2024-02-01,11,10,5
51,2024-03-01,11,9,5
52,2024-04-01,10,9,6
53,2024-05-01,11,11,6


#### Final City selection

from the above analysis we can see some city should be removed from our final analysis

Kuwait City: mean PM2.5 is negative, which is physically impossible. so we exclude this.

Abu Dhabi: useful PM2.5 coverage ends around 2023-06, so it does not align with our timeframe 2024–2025 story.

Cape Town: good data, but it ends around 2024-03, before Dubai’s main available period.

In [17]:
# Final city selection after coverage audit

import sys
import importlib
from pathlib import Path

project_root = Path(".")
scripts_dir = project_root / "scripts"

if str(scripts_dir) not in sys.path:
    sys.path.append(str(scripts_dir))

import city_selection_utils

importlib.reload(city_selection_utils)

from city_selection_utils import create_final_city_lists

In [18]:
city_coverage_summary_path = (
    project_root
    / "data"
    / "processed"
    / "combined"
    / "city_coverage_selection_summary.csv"
)

city_selection_output_dir = (
    project_root / "data" / "processed" / "combined" / "city_selection"
)

In [19]:
selection_tables, output_paths = create_final_city_lists(
    city_coverage_summary_path=city_coverage_summary_path,
    output_dir=city_selection_output_dir,
    excluded_main_cities=[
        "Kuwait City",
        "Abu Dhabi",
        "Cape Town",
    ],
    particle_profile_min_pm10_months=6,
    keep_context_cities_in_main=True,
)

output_paths

{'final_city_selection_all': WindowsPath('data/processed/combined/city_selection/final_city_selection_all.csv'),
 'selected_cities_pm25_main': WindowsPath('data/processed/combined/city_selection/selected_cities_pm25_main.csv'),
 'selected_cities_particle_profile': WindowsPath('data/processed/combined/city_selection/selected_cities_particle_profile.csv'),
 'excluded_or_context_cities': WindowsPath('data/processed/combined/city_selection/excluded_or_context_cities.csv')}

In [20]:
final_city_selection_all = selection_tables["final_city_selection_all"]
selected_cities_pm25_main = selection_tables["selected_cities_pm25_main"]
selected_cities_particle_profile = selection_tables["selected_cities_particle_profile"]
excluded_or_context_cities = selection_tables["excluded_or_context_cities"]

print("Total cities:", len(final_city_selection_all))
print("Selected PM2.5 main cities:", len(selected_cities_pm25_main))
print("Selected particle-profile cities:", len(selected_cities_particle_profile))
print("Excluded / context cities:", len(excluded_or_context_cities))

Total cities: 18
Selected PM2.5 main cities: 15
Selected particle-profile cities: 9
Excluded / context cities: 3


In [ ]:
# Read the CSV
df = pd.read_csv("data/processed/combined/city_selection/selected_cities_pm25_main.csv")

# Show summary + the full table
print(f"\n✅ Loaded {len(df)} cities")
display(df)


✅ Loaded 15 cities


,city,country,city_group,first_month,last_month,months_with_any_data,usable_pm25_months,usable_pm10_months,mean_pm25_month_coverage_pct,median_pm25_month_coverage_pct,...,core_city_pm25,strong_city_pm25,particle_profile_city,recommended_use,city_key,excluded_from_main,final_pm25_main_city,final_particle_profile_city,final_city_role,selection_note
0,bangkok,NaN,NaN,2021-04-01,2025-12-01,55,51,44,91.094680,100.000000,...,True,True,True,Core + particle-profile analysis,bangkok,False,True,True,Main PM2.5 + particle-profile analysis,Selected for both the main PM2.5 comparison an...
1,london,United Kingdom,Policy / developed-city comparison,2021-05-01,2026-01-01,55,50,50,87.235861,100.000000,...,True,True,True,Core + particle-profile analysis,london,False,True,True,Main PM2.5 + particle-profile analysis,Selected for both the main PM2.5 comparison an...
2,beijing,China,Dense but policy-relevant Asian comparison,2020-01-01,2026-01-01,39,37,26,89.044468,100.000000,...,True,True,True,Core + particle-profile analysis,beijing,False,True,True,Main PM2.5 + particle-profile analysis,Selected for both the main PM2.5 comparison an...
3,delhi,NaN,NaN,2019-12-01,2025-12-01,44,36,36,78.902039,98.387097,...,True,True,True,Core + particle-profile analysis,delhi,False,True,True,Main PM2.5 + particle-profile analysis,Selected for both the main PM2.5 comparison an...
4,los_angeles,United States,Policy / developed-city comparison,2022-02-01,2025-12-01,34,28,7,82.019517,100.000000,...,True,True,True,Core + particle-profile analysis,los_angeles,False,True,True,Main PM2.5 + particle-profile analysis,Selected for both the main PM2.5 comparison an...
5,mexico_city,Mexico,Optional contrast,2020-10-01,2026-01-01,33,27,27,76.321277,90.322581,...,True,True,True,Core + particle-profile analysis,mexico_city,False,True,True,Main PM2.5 + particle-profile analysis,Selected for both the main PM2.5 comparison an...
6,jakarta,NaN,NaN,2023-12-01,2025-12-01,25,24,11,94.451613,100.000000,...,True,True,True,Core + particle-profile analysis,jakarta,False,True,True,Main PM2.5 + particle-profile analysis,Selected for both the main PM2.5 comparison an...
7,new_york,United States,Policy / developed-city comparison,2023-11-01,2026-01-01,27,24,10,89.740570,100.000000,...,True,True,True,Core + particle-profile analysis,new_york,False,True,True,Main PM2.5 + particle-profile analysis,Selected for both the main PM2.5 comparison an...
8,seoul,South Korea,Dense but policy-relevant Asian comparison,2024-02-01,2026-01-01,24,22,21,91.013163,100.000000,...,True,True,True,Core + particle-profile analysis,seoul,False,True,True,Main PM2.5 + particle-profile analysis,Selected for both the main PM2.5 comparison an...
9,singapore,Singapore,Dense but policy-relevant Asian comparison,2022-10-01,2026-01-01,40,37,4,88.700851,100.000000,...,True,True,False,Core PM2.5 comparison,singapore,False,True,False,Main PM2.5 comparison,Selected for the main PM2.5 global comparison.


Fixing country name

In [28]:
# Fix missing country and city_group in final city-selection CSV files

from pathlib import Path
import pandas as pd
import re

city_selection_dir = Path("data/processed/combined/city_selection")

files_to_fix = [
    "final_city_selection_all.csv",
    "selected_cities_pm25_main.csv",
    "selected_cities_particle_profile.csv",
    "excluded_or_context_cities.csv",
]

city_to_country = {
    "dubai": "United Arab Emirates",
    "abu_dhabi": "United Arab Emirates",
    "riyadh": "Saudi Arabia",
    "doha": "Qatar",
    "kuwait_city": "Kuwait",
    "delhi": "India",
    "lahore": "Pakistan",
    "dhaka": "Bangladesh",
    "bangkok": "Thailand",
    "jakarta": "Indonesia",
    "singapore": "Singapore",
    "seoul": "South Korea",
    "tokyo": "Japan",
    "beijing": "China",
    "london": "United Kingdom",
    "los_angeles": "United States",
    "new_york": "United States",
    "mexico_city": "Mexico",
    "cape_town": "South Africa",
}

city_to_group = {
    "dubai": "Gulf / desert urbanization",
    "abu_dhabi": "Gulf / desert urbanization",
    "riyadh": "Gulf / desert urbanization",
    "doha": "Gulf / desert urbanization",
    "kuwait_city": "Gulf / desert urbanization",
    "delhi": "High-pollution / fast-growing Asia",
    "lahore": "High-pollution / fast-growing Asia",
    "dhaka": "High-pollution / fast-growing Asia",
    "bangkok": "High-pollution / fast-growing Asia",
    "jakarta": "High-pollution / fast-growing Asia",
    "singapore": "Dense but policy-relevant Asian comparison",
    "seoul": "Dense but policy-relevant Asian comparison",
    "tokyo": "Dense but policy-relevant Asian comparison",
    "beijing": "Dense but policy-relevant Asian comparison",
    "london": "Policy / developed-city comparison",
    "los_angeles": "Policy / developed-city comparison",
    "new_york": "Policy / developed-city comparison",
    "mexico_city": "Optional contrast",
    "cape_town": "Optional contrast",
}


def normalize_city_name(value):
    # Stable city-name key for both "New York" and "new_york"
    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value


def is_missing_text(series):
    # Missing text detector for NaN, empty strings, and string "nan"
    return (
        series.isna()
        | (series.astype(str).str.strip() == "")
        | (series.astype(str).str.lower().str.strip() == "nan")
        | (series.astype(str).str.lower().str.strip() == "none")
    )


def fix_city_metadata(df):
    # Fill missing country and city_group from city name
    df = df.copy()

    if "city" not in df.columns:
        raise ValueError("city column is missing.")

    df["city_key"] = df["city"].apply(normalize_city_name)

    if "country" not in df.columns:
        df["country"] = pd.NA

    if "city_group" not in df.columns:
        df["city_group"] = pd.NA

    missing_country = is_missing_text(df["country"])
    missing_city_group = is_missing_text(df["city_group"])

    df.loc[missing_country, "country"] = df.loc[missing_country, "city_key"].map(
        city_to_country
    )

    df.loc[missing_city_group, "city_group"] = df.loc[
        missing_city_group, "city_key"
    ].map(city_to_group)

    # Optional cleaner display names for city column
    city_display_names = {
        "dubai": "Dubai",
        "abu_dhabi": "Abu Dhabi",
        "riyadh": "Riyadh",
        "doha": "Doha",
        "kuwait_city": "Kuwait City",
        "delhi": "Delhi",
        "lahore": "Lahore",
        "dhaka": "Dhaka",
        "bangkok": "Bangkok",
        "jakarta": "Jakarta",
        "singapore": "Singapore",
        "seoul": "Seoul",
        "tokyo": "Tokyo",
        "beijing": "Beijing",
        "london": "London",
        "los_angeles": "Los Angeles",
        "new_york": "New York",
        "mexico_city": "Mexico City",
        "cape_town": "Cape Town",
    }

    df["city_display"] = df["city_key"].map(city_display_names).fillna(df["city"])

    return df


for file_name in files_to_fix:
    path = city_selection_dir / file_name

    if not path.exists():
        print(f"Skipped missing file: {path}")
        continue

    df = pd.read_csv(path)
    fixed_df = fix_city_metadata(df)

    fixed_df.to_csv(path, index=False)

    print(f"Rewritten: {path}")
    print("Missing country after fix:", fixed_df["country"].isna().sum())
    print("Missing city_group after fix:", fixed_df["city_group"].isna().sum())
    print()


# Quick check
selected_path = city_selection_dir / "selected_cities_pm25_main.csv"
selected_cities_pm25_main = pd.read_csv(selected_path)

selected_cities_pm25_main[
    [
        "city",
        "city_display",
        "country",
        "city_group",
        "usable_pm25_months",
        "usable_pm10_months",
        "final_city_role",
    ]
]

Rewritten: data\processed\combined\city_selection\final_city_selection_all.csv
Missing country after fix: 0
Missing city_group after fix: 0

Rewritten: data\processed\combined\city_selection\selected_cities_pm25_main.csv
Missing country after fix: 0
Missing city_group after fix: 0

Rewritten: data\processed\combined\city_selection\selected_cities_particle_profile.csv
Missing country after fix: 0
Missing city_group after fix: 0

Rewritten: data\processed\combined\city_selection\excluded_or_context_cities.csv
Missing country after fix: 0
Missing city_group after fix: 0



,city,city_display,country,city_group,usable_pm25_months,usable_pm10_months,final_city_role
0,bangkok,Bangkok,Thailand,High-pollution / fast-growing Asia,51,44,Main PM2.5 + particle-profile analysis
1,london,London,United Kingdom,Policy / developed-city comparison,50,50,Main PM2.5 + particle-profile analysis
2,beijing,Beijing,China,Dense but policy-relevant Asian comparison,37,26,Main PM2.5 + particle-profile analysis
3,delhi,Delhi,India,High-pollution / fast-growing Asia,36,36,Main PM2.5 + particle-profile analysis
4,los_angeles,Los Angeles,United States,Policy / developed-city comparison,28,7,Main PM2.5 + particle-profile analysis
5,mexico_city,Mexico City,Mexico,Optional contrast,27,27,Main PM2.5 + particle-profile analysis
6,jakarta,Jakarta,Indonesia,High-pollution / fast-growing Asia,24,11,Main PM2.5 + particle-profile analysis
7,new_york,New York,United States,Policy / developed-city comparison,24,10,Main PM2.5 + particle-profile analysis
8,seoul,Seoul,South Korea,Dense but policy-relevant Asian comparison,22,21,Main PM2.5 + particle-profile analysis
9,singapore,Singapore,Singapore,Dense but policy-relevant Asian comparison,37,4,Main PM2.5 comparison


#### Adding weather data to our openAQ data to make it more feasible and enrich for our data analysis. 

In [ ]:


import sys
import importlib
from pathlib import Path

project_root = Path(".")
scripts_dir = project_root / "scripts"

if str(scripts_dir) not in sys.path:
    sys.path.append(str(scripts_dir))

import weather_enrichment_by_city

importlib.reload(weather_enrichment_by_city)


def find_city_file_csv_first(city_folder, city_slug, table_name):
    # City file finder that prefers CSV to avoid PyArrow / Parquet issues
    city_folder = Path(city_folder)

    csv_path = city_folder / f"{city_slug}_{table_name}.csv"
    parquet_path = city_folder / f"{city_slug}_{table_name}.parquet"

    if csv_path.exists():
        return csv_path

    if parquet_path.exists():
        return parquet_path

    raise FileNotFoundError(
        f"Could not find {table_name} for {city_slug} in {city_folder}"
    )


# Patch the script function inside the current notebook session
weather_enrichment_by_city.find_city_file = find_city_file_csv_first

from weather_enrichment_by_city import process_selected_cities_weather

config

In [37]:
weather_result = process_selected_cities_weather(
    selected_cities_path="data/processed/combined/city_selection/selected_cities_pm25_main.csv",
    input_root="data/processed/openaq_by_city",
    output_root="data/processed/enriched",
    start_date="2025-01-01",
    end_date="2025-12-31",
    output_format="csv",
    overwrite_weather=False,
    max_cities=None,
    include_hourly_combined=False,
)

weather_result["log"]

[1/15] Processing weather for bangkok
[2/15] Processing weather for london
[3/15] Processing weather for beijing
[4/15] Processing weather for delhi
[5/15] Processing weather for los_angeles
[6/15] Processing weather for mexico_city
[7/15] Processing weather for jakarta
[8/15] Processing weather for new_york
[9/15] Processing weather for seoul
[10/15] Processing weather for singapore
[11/15] Processing weather for tokyo
[12/15] Processing weather for lahore
[13/15] Processing weather for dubai
[14/15] Processing weather for riyadh
[15/15] Processing weather for dhaka


,city_slug,city,country,start_date,end_date,weather_latitude,weather_longitude,city_hourly_rows,weather_hourly_rows,city_hourly_enriched_rows,city_daily_enriched_rows,city_monthly_enriched_rows,output_folder,status,error
0,bangkok,bangkok,NaN,2025-01-01,2025-12-31,13.703342,100.485100,8094,8760,8094,365,12,data\processed\enriched\by_city\bangkok,success,None
1,london,london,United Kingdom,2025-01-01,2025-12-31,51.490610,-0.090745,8755,8760,8755,365,12,data\processed\enriched\by_city\london,success,None
2,beijing,beijing,China,2025-02-11,2025-12-31,39.937743,116.443455,7629,7776,7629,323,12,data\processed\enriched\by_city\beijing,success,None
3,delhi,delhi,NaN,2025-02-18,2025-12-31,28.731082,77.234521,7298,7608,7298,316,11,data\processed\enriched\by_city\delhi,success,None
4,los_angeles,los_angeles,United States,2025-01-01,2025-12-31,34.277280,-118.456810,7327,8760,7327,366,13,data\processed\enriched\by_city\los_angeles,success,None
5,mexico_city,mexico_city,Mexico,2025-01-01,2025-12-31,19.400694,-99.204722,7979,8760,7979,354,13,data\processed\enriched\by_city\mexico_city,success,None
6,jakarta,jakarta,NaN,2025-01-01,2025-12-31,-6.409456,106.856152,8099,8760,8099,355,12,data\processed\enriched\by_city\jakarta,success,None
7,new_york,new_york,United States,2025-01-01,2025-12-31,40.740022,-74.206574,7923,8760,7923,337,13,data\processed\enriched\by_city\new_york,success,None
8,seoul,seoul,South Korea,2025-01-01,2025-12-31,37.546111,127.002329,8721,8760,8721,366,13,data\processed\enriched\by_city\seoul,success,None
9,singapore,singapore,Singapore,2025-01-01,2025-12-31,1.337797,103.845914,7924,8760,7924,337,13,data\processed\enriched\by_city\singapore,success,None


keeping only local year

In [38]:
# Clean enriched combined files to the 2025 analysis window

from pathlib import Path
import pandas as pd

enriched_combined_dir = Path("data/processed/enriched/combined")

daily_path = enriched_combined_dir / "combined_city_daily_enriched.csv"
monthly_path = enriched_combined_dir / "combined_city_monthly_enriched.csv"

daily = pd.read_csv(daily_path)
monthly = pd.read_csv(monthly_path)

# Keep only local calendar year 2025
daily_2025 = daily[daily["local_year"] == 2025].copy()
monthly_2025 = monthly[monthly["local_year"] == 2025].copy()

daily_2025_path = enriched_combined_dir / "combined_city_daily_enriched_2025.csv"
monthly_2025_path = enriched_combined_dir / "combined_city_monthly_enriched_2025.csv"

daily_2025.to_csv(daily_2025_path, index=False)
monthly_2025.to_csv(monthly_2025_path, index=False)

print("Saved:")
print(daily_2025_path)
print(monthly_2025_path)

print("\nDaily rows:", len(daily_2025))
print("Monthly rows:", len(monthly_2025))

monthly_2025.groupby("city").agg(
    months=("local_month", "nunique"),
    first_month=("local_month", "min"),
    last_month=("local_month", "max"),
    valid_pm25_days=("valid_pm25_days", "sum"),
    valid_pm10_days=("valid_pm10_days", "sum"),
).reset_index()

Saved:
data\processed\enriched\combined\combined_city_daily_enriched_2025.csv
data\processed\enriched\combined\combined_city_monthly_enriched_2025.csv

Daily rows: 5103
Monthly rows: 178


,city,months,first_month,last_month,valid_pm25_days,valid_pm10_days
0,bangkok,12,1,12,365,261
1,beijing,11,2,12,322,0
2,delhi,11,2,12,316,316
3,dhaka,12,1,12,238,13
4,dubai,12,1,12,337,13
5,jakarta,12,1,12,355,13
6,lahore,12,1,12,365,0
7,london,12,1,12,365,365
8,los_angeles,12,1,12,365,0
9,mexico_city,12,1,12,353,353


In [ ]:
# ============================================================
#  PM2.5 quality check: mean vs median
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
import plotly.express as px

combined_enriched_dir = Path("data/processed/enriched/combined")
web_data_output_dir = Path("web/data")
figure_output_dir = Path("outputs/figures")

web_data_output_dir.mkdir(parents=True, exist_ok=True)
figure_output_dir.mkdir(parents=True, exist_ok=True)

daily = pd.read_csv(combined_enriched_dir / "combined_city_daily_enriched_2025.csv")
monthly = pd.read_csv(combined_enriched_dir / "combined_city_monthly_enriched_2025.csv")
weather_points = pd.read_csv(combined_enriched_dir / "combined_city_weather_points.csv")


def clean_city_key(value):
    return str(value).strip().lower().replace(" ", "_").replace("-", "_")


city_display_lookup = {
    "dubai": "Dubai",
    "riyadh": "Riyadh",
    "delhi": "Delhi",
    "lahore": "Lahore",
    "dhaka": "Dhaka",
    "bangkok": "Bangkok",
    "jakarta": "Jakarta",
    "singapore": "Singapore",
    "seoul": "Seoul",
    "tokyo": "Tokyo",
    "beijing": "Beijing",
    "london": "London",
    "los_angeles": "Los Angeles",
    "new_york": "New York",
    "mexico_city": "Mexico City",
}

for df in [daily, monthly, weather_points]:
    df["city_key"] = df["city"].apply(clean_city_key)
    df["city_display"] = df["city_key"].map(city_display_lookup).fillna(df["city"])

monthly["month_date"] = pd.to_datetime(
    monthly["local_year"].astype(int).astype(str)
    + "-"
    + monthly["local_month"].astype(int).astype(str).str.zfill(2)
    + "-01"
)

monthly["month_label"] = monthly["month_date"].dt.strftime("%b %Y")
daily["local_date"] = pd.to_datetime(daily["local_date"], errors="coerce")

# Basic impossible-value cleanup flags
for col in [
    "pm25_monthly_mean",
    "pm25_monthly_median",
    "pm25_monthly_p90",
    "pm10_monthly_mean",
    "pm10_monthly_median",
    "pm10_monthly_p90",
]:
    if col in monthly.columns:
        monthly.loc[monthly[col] < 0, col] = np.nan

for col in [
    "pm25_daily_mean",
    "pm25_daily_median",
    "pm25_daily_p90",
    "pm10_daily_mean",
    "pm10_daily_median",
    "pm10_daily_p90",
]:
    if col in daily.columns:
        daily.loc[daily[col] < 0, col] = np.nan

# Mean-median distortion check
monthly["pm25_mean_median_ratio"] = monthly["pm25_monthly_mean"] / monthly[
    "pm25_monthly_median"
].replace(0, np.nan)

monthly["pm25_mean_minus_median"] = (
    monthly["pm25_monthly_mean"] - monthly["pm25_monthly_median"]
)

suspicious_months = monthly[
    (
        (monthly["pm25_mean_median_ratio"] >= 3)
        & (monthly["pm25_mean_minus_median"] >= 20)
    )
    | (monthly["pm25_monthly_p90"] >= 200)
].copy()

suspicious_months = suspicious_months.sort_values(
    ["pm25_monthly_p90", "pm25_mean_median_ratio"], ascending=[False, False]
)

suspicious_months[
    [
        "city_display",
        "month_label",
        "pm25_monthly_mean",
        "pm25_monthly_median",
        "pm25_monthly_p90",
        "pm25_mean_median_ratio",
        "pm25_mean_minus_median",
        "valid_pm25_days",
        "pm25_month_coverage_pct",
    ]
].head(30)

,city_display,month_label,pm25_monthly_mean,pm25_monthly_median,pm25_monthly_p90,pm25_mean_median_ratio,pm25_mean_minus_median,valid_pm25_days,pm25_month_coverage_pct
152,Seoul,Nov 2025,799.725694,21.750000,3113.089583,36.768997,777.975694,30,100.000000
151,Seoul,Oct 2025,680.865762,7.500000,2917.208333,90.782102,673.365762,31,100.000000
144,Seoul,Mar 2025,152.462366,26.000000,436.750000,5.863937,126.462366,31,100.000000
81,Lahore,Dec 2025,250.856879,251.431251,325.238735,0.997716,-0.574372,31,100.000000
34,Dhaka,Jan 2025,226.604306,217.483707,302.305429,1.041937,9.120599,13,41.935484
33,Delhi,Dec 2025,212.468625,187.500000,288.618750,1.133166,24.968625,31,100.000000
32,Delhi,Nov 2025,229.166320,225.993750,284.755000,1.014038,3.172570,30,100.000000
80,Lahore,Nov 2025,194.641467,215.051620,263.503253,0.905092,-20.410153,30,100.000000
35,Dhaka,Feb 2025,188.558156,195.678032,235.983785,0.963614,-7.119877,18,64.285714
70,Lahore,Jan 2025,181.620087,182.347500,230.631176,0.996011,-0.727413,31,100.000000


we can see in Seol there is some problem in PM2.5 values. This may occured due to sensor error or mismatch. so we have to handle this issue.

In [2]:
# ============================================================
# Data quality fix: remove invalid PM sentinel values and rebuild enriched data
# ============================================================

import sys
import importlib
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path(".")
scripts_dir = project_root / "scripts"

if str(scripts_dir) not in sys.path:
    sys.path.append(str(scripts_dir))

import weather_enrichment_by_city

importlib.reload(weather_enrichment_by_city)

from weather_enrichment_by_city import (
    aggregate_city_daily_enriched,
    aggregate_city_monthly_enriched,
)

# Existing enriched data location
enriched_root = Path("data/processed/enriched")
enriched_by_city_dir = enriched_root / "by_city"
enriched_combined_dir = enriched_root / "combined"

# One-time backup before overwriting
backup_root = Path("data/processed/enriched_backup_before_pm_cleaning")
backup_root.mkdir(parents=True, exist_ok=True)

BACKUP_ORIGINAL_FILES = True

# Invalid-value rules
# 9999 / 5000+ values are almost certainly missing/error sentinel values.
PM25_MIN_VALID = 0
PM25_MAX_VALID = 1000

PM10_MIN_VALID = 0
PM10_MAX_VALID = 2000


def backup_file(path):
    # Back up a file before overwriting it
    path = Path(path)

    if not BACKUP_ORIGINAL_FILES or not path.exists():
        return

    relative_path = path.relative_to(enriched_root)
    backup_path = backup_root / relative_path
    backup_path.parent.mkdir(parents=True, exist_ok=True)

    if not backup_path.exists():
        shutil.copy2(path, backup_path)


def clean_pm_columns(df):
    # Remove invalid PM values from city-hourly enriched data
    df = df.copy()

    pm25_columns = [
        "pm25_median",
        "pm25_mean",
        "pm25_p90",
        "pm25_min",
        "pm25_max",
    ]

    pm10_columns = [
        "pm10_median",
        "pm10_mean",
        "pm10_p90",
        "pm10_min",
        "pm10_max",
    ]

    invalid_counts = {}

    for col in pm25_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

            invalid_mask = (df[col] < PM25_MIN_VALID) | (df[col] > PM25_MAX_VALID)

            invalid_counts[f"invalid_{col}"] = int(invalid_mask.sum())
            df.loc[invalid_mask, col] = np.nan

    for col in pm10_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

            invalid_mask = (df[col] < PM10_MIN_VALID) | (df[col] > PM10_MAX_VALID)

            invalid_counts[f"invalid_{col}"] = int(invalid_mask.sum())
            df.loc[invalid_mask, col] = np.nan

    # If the main city-hour PM value is invalid, the valid-location count should not count it
    if "valid_pm25_locations" in df.columns and "pm25_median" in df.columns:
        df.loc[df["pm25_median"].isna(), "valid_pm25_locations"] = 0

    if "valid_pm10_locations" in df.columns and "pm10_median" in df.columns:
        df.loc[df["pm10_median"].isna(), "valid_pm10_locations"] = 0

    # Recalculate PM2.5 / PM10 ratio only when both medians are valid
    if {"pm25_median", "pm10_median"}.issubset(df.columns):
        df["pm25_pm10_ratio_from_city_medians"] = np.where(
            (df["pm25_median"].notna())
            & (df["pm10_median"].notna())
            & (df["pm10_median"] > 0),
            df["pm25_median"] / df["pm10_median"],
            np.nan,
        )

    # Remove impossible ratio values if present
    for ratio_col in [
        "pm25_pm10_ratio_median",
        "pm25_pm10_ratio_mean",
        "pm25_pm10_ratio_from_city_medians",
    ]:
        if ratio_col in df.columns:
            df[ratio_col] = pd.to_numeric(df[ratio_col], errors="coerce")
            invalid_ratio = (df[ratio_col] < 0) | (df[ratio_col] > 10)
            invalid_counts[f"invalid_{ratio_col}"] = int(invalid_ratio.sum())
            df.loc[invalid_ratio, ratio_col] = np.nan

    return df, invalid_counts


def save_csv_overwrite(df, path):
    # Save CSV after backing up old file
    path = Path(path)
    backup_file(path)
    df.to_csv(path, index=False)


def remove_stale_parquet_if_exists(csv_path):
    # Avoid accidentally reading stale Parquet files after CSV cleaning
    csv_path = Path(csv_path)
    parquet_path = csv_path.with_suffix(".parquet")

    if parquet_path.exists():
        backup_file(parquet_path)

        stale_path = parquet_path.with_suffix(".parquet.stale_before_pm_cleaning")

        if not stale_path.exists():
            parquet_path.rename(stale_path)


city_hourly_files = sorted(enriched_by_city_dir.glob("*/*_city_hourly_enriched.csv"))

print("City hourly enriched files found:", len(city_hourly_files))

if len(city_hourly_files) == 0:
    raise FileNotFoundError("No city hourly enriched CSV files found.")

clean_daily_frames = []
clean_monthly_frames = []
cleaning_logs = []

for hourly_path in city_hourly_files:
    city_slug = hourly_path.parent.name

    print(f"Cleaning {city_slug}...")

    hourly = pd.read_csv(hourly_path)

    hourly_clean, invalid_counts = clean_pm_columns(hourly)

    daily_clean = aggregate_city_daily_enriched(
        city_hourly_enriched=hourly_clean,
        min_valid_hours=12,
    )

    monthly_clean = aggregate_city_monthly_enriched(
        city_daily_enriched=daily_clean,
        min_valid_days=15,
    )

    city_output_dir = hourly_path.parent

    hourly_clean_path = city_output_dir / f"{city_slug}_city_hourly_enriched.csv"
    daily_clean_path = city_output_dir / f"{city_slug}_city_daily_enriched.csv"
    monthly_clean_path = city_output_dir / f"{city_slug}_city_monthly_enriched.csv"

    save_csv_overwrite(hourly_clean, hourly_clean_path)
    save_csv_overwrite(daily_clean, daily_clean_path)
    save_csv_overwrite(monthly_clean, monthly_clean_path)

    # Prevent stale Parquet from being used accidentally
    remove_stale_parquet_if_exists(hourly_clean_path)
    remove_stale_parquet_if_exists(daily_clean_path)
    remove_stale_parquet_if_exists(monthly_clean_path)

    clean_daily_frames.append(daily_clean)
    clean_monthly_frames.append(monthly_clean)

    log_row = {
        "city_slug": city_slug,
        "hourly_rows": len(hourly),
        "daily_rows_after_cleaning": len(daily_clean),
        "monthly_rows_after_cleaning": len(monthly_clean),
    }

    log_row.update(invalid_counts)
    cleaning_logs.append(log_row)


# Rebuild combined enriched daily/monthly files
combined_daily_clean = pd.concat(clean_daily_frames, ignore_index=True)
combined_monthly_clean = pd.concat(clean_monthly_frames, ignore_index=True)
cleaning_log = pd.DataFrame(cleaning_logs)

# Keep only local calendar year 2025
combined_daily_clean = combined_daily_clean[
    combined_daily_clean["local_year"] == 2025
].copy()

combined_monthly_clean = combined_monthly_clean[
    combined_monthly_clean["local_year"] == 2025
].copy()

combined_daily_path = enriched_combined_dir / "combined_city_daily_enriched.csv"
combined_monthly_path = enriched_combined_dir / "combined_city_monthly_enriched.csv"

combined_daily_2025_path = (
    enriched_combined_dir / "combined_city_daily_enriched_2025.csv"
)
combined_monthly_2025_path = (
    enriched_combined_dir / "combined_city_monthly_enriched_2025.csv"
)

cleaning_log_path = enriched_combined_dir / "pm_invalid_value_cleaning_log.csv"

save_csv_overwrite(combined_daily_clean, combined_daily_path)
save_csv_overwrite(combined_monthly_clean, combined_monthly_path)

save_csv_overwrite(combined_daily_clean, combined_daily_2025_path)
save_csv_overwrite(combined_monthly_clean, combined_monthly_2025_path)

cleaning_log.to_csv(cleaning_log_path, index=False)

remove_stale_parquet_if_exists(combined_daily_path)
remove_stale_parquet_if_exists(combined_monthly_path)
remove_stale_parquet_if_exists(combined_daily_2025_path)
remove_stale_parquet_if_exists(combined_monthly_2025_path)

print("\nSaved fixed files:")
print(combined_daily_2025_path)
print(combined_monthly_2025_path)
print(cleaning_log_path)

print("\nBackup folder:")
print(backup_root)

cleaning_log.sort_values(
    (
        [col for col in cleaning_log.columns if col.startswith("invalid_pm25_median")][
            0
        ]
        if any(col.startswith("invalid_pm25_median") for col in cleaning_log.columns)
        else "city_slug"
    ),
    ascending=False,
).head(20)

City hourly enriched files found: 15
Cleaning bangkok...
Cleaning beijing...
Cleaning delhi...
Cleaning dhaka...
Cleaning dubai...
Cleaning jakarta...
Cleaning lahore...
Cleaning london...
Cleaning los_angeles...
Cleaning mexico_city...
Cleaning new_york...
Cleaning riyadh...
Cleaning seoul...
Cleaning singapore...
Cleaning tokyo...

Saved fixed files:
data\processed\enriched\combined\combined_city_daily_enriched_2025.csv
data\processed\enriched\combined\combined_city_monthly_enriched_2025.csv
data\processed\enriched\combined\pm_invalid_value_cleaning_log.csv

Backup folder:
data\processed\enriched_backup_before_pm_cleaning


,city_slug,hourly_rows,daily_rows_after_cleaning,monthly_rows_after_cleaning,invalid_pm25_median,invalid_pm25_mean,invalid_pm25_p90,invalid_pm25_min,invalid_pm25_max,invalid_pm10_median,invalid_pm10_mean,invalid_pm10_p90,invalid_pm10_min,invalid_pm10_max,invalid_pm25_pm10_ratio_median,invalid_pm25_pm10_ratio_mean,invalid_pm25_pm10_ratio_from_city_medians
12,seoul,8721,366,13,153,2029,2029,0,2029,111,1780,1780,1,1780,28,309,0
2,delhi,7298,316,11,1,2,19,0,25,9,15,24,10,36,1,6,0
0,bangkok,8094,365,12,0,0,0,0,0,0,0,0,0,0,0,0,0
3,dhaka,5181,238,12,0,0,0,0,0,0,0,0,0,0,0,0,0
4,dubai,7920,337,12,0,0,0,0,0,0,0,0,0,0,0,0,0
5,jakarta,8099,355,12,0,0,0,0,0,0,0,0,0,0,0,0,0
1,beijing,7629,323,12,0,0,0,0,0,0,0,0,0,0,0,0,0
6,lahore,6610,365,12,0,0,0,0,0,0,0,0,0,0,0,0,0
7,london,8755,365,12,0,0,0,3,0,1,2,0,18,0,0,0,0
9,mexico_city,7979,354,13,0,0,0,0,0,0,0,0,0,0,1,2,1


In [3]:
# ============================================================
# Check Seoul after cleaning
# ============================================================

daily_fixed = pd.read_csv(
    "data/processed/enriched/combined/combined_city_daily_enriched_2025.csv"
)
monthly_fixed = pd.read_csv(
    "data/processed/enriched/combined/combined_city_monthly_enriched_2025.csv"
)


def city_key(value):
    return str(value).strip().lower().replace(" ", "_").replace("-", "_")


daily_fixed["city_key"] = daily_fixed["city"].apply(city_key)
monthly_fixed["city_key"] = monthly_fixed["city"].apply(city_key)

seoul_daily = daily_fixed[daily_fixed["city_key"] == "seoul"].copy()
seoul_monthly = monthly_fixed[monthly_fixed["city_key"] == "seoul"].copy()

print("Seoul daily rows:", len(seoul_daily))
print("Seoul monthly rows:", len(seoul_monthly))

display(
    seoul_monthly[
        [
            "city",
            "local_year",
            "local_month",
            "pm25_monthly_mean",
            "pm25_monthly_median",
            "pm25_monthly_p90",
            "valid_pm25_days",
            "pm25_month_coverage_pct",
            "pm25_month_is_usable",
        ]
    ]
)

display(
    seoul_daily.sort_values("pm25_daily_mean", ascending=False)[
        [
            "city",
            "local_date",
            "pm25_daily_mean",
            "pm25_daily_median",
            "pm25_daily_p90",
            "pm25_daily_max",
            "valid_pm25_hours",
            "pm25_daily_coverage_pct",
        ]
    ].head(20)
)

Seoul daily rows: 365
Seoul monthly rows: 12


,city,local_year,local_month,pm25_monthly_mean,pm25_monthly_median,pm25_monthly_p90,valid_pm25_days,pm25_month_coverage_pct,pm25_month_is_usable
142,seoul,2025,1,27.681183,21.50,48.333333,31,100.0,True
143,seoul,2025,2,21.913043,16.75,36.408333,28,100.0,True
144,seoul,2025,3,31.642997,26.00,52.333333,31,100.0,True
145,seoul,2025,4,23.450670,22.75,34.233333,30,100.0,True
146,seoul,2025,5,19.506235,18.50,31.500000,31,100.0,True
147,seoul,2025,6,17.133081,17.25,26.366667,30,100.0,True
148,seoul,2025,7,11.668355,11.00,19.500000,31,100.0,True
149,seoul,2025,8,9.632209,8.00,15.166667,31,100.0,True
150,seoul,2025,9,7.234058,6.50,12.783333,30,100.0,True
151,seoul,2025,10,9.050625,6.50,17.000000,31,100.0,True


,city,local_date,pm25_daily_mean,pm25_daily_median,pm25_daily_p90,pm25_daily_max,valid_pm25_hours,pm25_daily_coverage_pct
4057,seoul,2025-01-21,84.458333,87.0,92.70,100.0,24,100.000000
4120,seoul,2025-03-25,66.791667,73.0,86.70,88.0,24,100.000000
4056,seoul,2025-01-20,65.583333,60.5,93.10,102.0,24,100.000000
4365,seoul,2025-11-25,62.173913,66.0,78.90,86.0,23,95.833333
4058,seoul,2025-01-22,61.541667,57.0,76.00,79.0,24,100.000000
4364,seoul,2025-11-24,58.431818,52.0,84.85,92.5,22,91.666667
4105,seoul,2025-03-10,57.375000,56.5,66.40,73.0,24,100.000000
4116,seoul,2025-03-21,55.750000,58.5,71.80,74.0,24,100.000000
4117,seoul,2025-03-22,52.333333,49.0,69.70,74.0,24,100.000000
4083,seoul,2025-02-16,52.333333,42.5,79.10,86.0,24,100.000000


In [ ]:

# Cleaning city summary after PM invalid-value removal


monthly_fixed["city_key"] = monthly_fixed["city"].apply(city_key)

clean_city_summary = (
    monthly_fixed.groupby("city_key")
    .agg(
        months_with_data=("local_month", "nunique"),
        usable_pm25_months=(
            "pm25_month_is_usable",
            lambda x: int(pd.Series(x).fillna(False).astype(bool).sum()),
        ),
        usable_pm10_months=(
            "pm10_month_is_usable",
            lambda x: int(pd.Series(x).fillna(False).astype(bool).sum()),
        ),
        annual_median_pm25=("pm25_monthly_median", "median"),
        annual_mean_pm25=("pm25_monthly_mean", "mean"),
        annual_p90_pm25=("pm25_monthly_p90", "median"),
        annual_median_pm10=("pm10_monthly_median", "median"),
        annual_mean_pm10=("pm10_monthly_mean", "mean"),
        mean_pm25_coverage=("pm25_month_coverage_pct", "mean"),
        mean_pm10_coverage=("pm10_month_coverage_pct", "mean"),
    )
    .reset_index()
    .sort_values("annual_median_pm25", ascending=False)
)

clean_city_summary.to_csv(
    enriched_combined_dir / "clean_city_summary_2025.csv",
    index=False,
)

clean_city_summary

,city_key,months_with_data,usable_pm25_months,usable_pm10_months,annual_median_pm25,annual_mean_pm25,annual_p90_pm25,annual_median_pm10,annual_mean_pm10,mean_pm25_coverage,mean_pm10_coverage
2,delhi,11,10,10,73.500000,95.976970,122.489583,179.250000,215.935941,94.177489,94.177489
3,dhaka,12,9,0,68.458743,99.124929,100.240185,243.385688,248.661824,65.222734,3.494624
6,lahore,12,12,0,49.796250,97.357319,78.231087,NaN,NaN,100.000000,0.000000
5,jakarta,12,12,0,46.284989,45.610950,63.385363,45.260714,46.534551,97.311828,3.494624
4,dubai,12,11,0,34.298356,37.290244,59.355745,38.639875,39.153569,92.185100,3.494624
1,beijing,11,11,0,30.438139,34.938278,62.057083,NaN,NaN,96.135316,0.000000
9,mexico_city,12,12,12,18.937500,19.821132,27.290138,39.062500,37.552060,96.698669,96.698669
11,riyadh,12,10,0,18.868323,19.234812,30.758049,37.409917,33.672973,87.615207,3.494624
12,seoul,12,12,12,17.000000,18.558680,32.010417,30.625000,34.643133,100.000000,99.722222
13,singapore,12,11,0,16.834392,19.064298,28.521481,15.520677,16.112965,91.887481,3.494624
